In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive # type:ignore
drive.mount('/content/drive')

import shutil
shutil.copytree("/content/drive/MyDrive/Datasets/Cats vs Dogs", "/content/cats_vs_dogs", dirs_exist_ok=True)

In [ ]:
import tensorflow as tf # type:ignore
from tensorflow.keras import Sequential # type:ignore
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPooling2D, BatchNormalization, Dropout # type:ignore

In [ ]:
trainx = "/content/drive/MyDrive/Datasets/Cats vs Dogs/train"
testx = "/content/drive/MyDrive/Datasets/Cats vs Dogs/test"

In [ ]:
# Generators: Used to convert data in batches so RAM can process easily. Also it used to rescale the image on the same scale.

def ImageTransformation(trainx,testx):

    train_generator = tf.keras.utils.image_dataset_from_directory(
        directory = trainx,
        labels = "inferred",
        label_mode = "int", # Assigning 0 to cats and 1 to dogs
        batch_size = 33,
        image_size = (180,180)
    )

    test_generator = tf.keras.utils.image_dataset_from_directory(
        directory = testx,
        labels = "inferred",
        label_mode = "int", # Assigning 0 to cats and 1 to dogs
        batch_size = 33,
        image_size = (180,180)
    )

    # Normalization: Every val in numpy array is between 0 and 255, we will convert it between 0 and 1.

    def Transformation(img, label):
        img = tf.cast(img/255, tf.float32)
        return img,label

    train_pixels = train_generator.map(Transformation)
    test_pixels = test_generator.map(Transformation)

    return train_pixels, test_pixels

train_pixels, test_pixels = ImageTransformation(trainx, testx)

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# CNN Model:

m = Sequential()

m.add(Conv2D(32, kernel_size=(3,3), padding="same", activation="relu", input_shape = (180,180,3)))
m.add(BatchNormalization())
m.add(MaxPooling2D(pool_size=(2,2), padding = "valid", strides = 2))

m.add(Conv2D(64, kernel_size=(3,3), padding="same", activation="relu"))
m.add(BatchNormalization())
m.add(MaxPooling2D(pool_size=(2,2), padding = "valid", strides = 2))

m.add(Conv2D(128, kernel_size=(3,3), padding="same", activation="relu"))
m.add(BatchNormalization())
m.add(MaxPooling2D(pool_size=(2,2), padding = "valid", strides = 2))

m.add(Flatten())

m.add(Dense(128, activation="relu")) # First layer
m.add(Dropout(0.2))
m.add(Dense(64, activation="relu")) # Second layer
m.add(Dropout(0.2))
m.add(Dense(1, activation="sigmoid")) # Output layer

m.compile(loss="binary_crossentropy", optimizer="Adam", metrics=["accuracy"]) # Using Adam gradient descent as optimizer

m.fit(train_pixels, validation_data=test_pixels, epochs=20)


In [ ]:
history = m.history_

import matplotlib.pyplot as plt
plt.plot(history['accuracy'], label='train accuracy')
plt.plot(history['val_accuracy'], label='test accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()